# 1. Setup and Imports
Load all the necessary libraries.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from tqdm.notebook import tqdm
import json
from pathlib import Path

# 2. Configuration & Intent Labels
Define the chatbot intents and model hyperparameters.

In [ ]:
class Config:
    model_name: str = "xlm-roberta-base"
    max_length: int = 160
    batch_size: int = 16
    learning_rate: float = 2e-5
    epochs: int = 4
    seed: int = 42

config = Config()

INTENT_MAP = {
    0: "PRICE_INQUIRY",
    1: "PRODUCT_QUESTION",
    2: "DELIVERY_QUESTION",
    3: "OPENING_HOURS",
    4: "COMPLAINT",
    5: "DISCOUNT_REQUEST",
    6: "OTHER"
}


# 3. Dataset Loader
Define the PyTorch dataset to load `train.csv` and `eval.csv`.

In [ ]:
class ModerationDataset(Dataset):
    def __init__(self, csv_path: str, tokenizer, max_length: int):
        self.df = pd.read_csv(csv_path)
        if "text" not in self.df.columns or "intent_id" not in self.df.columns:
            raise ValueError(f"{csv_path} is missing required columns")
            
        self.texts = self.df["text"].astype(str).tolist()
        self.intent_ids = self.df["intent_id"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict:
        encoding = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "intent_id": torch.tensor(self.intent_ids[index], dtype=torch.long),
        }


# 4. Model Architecture
Single-head classification model for intent detection.

In [ ]:
class IntentModel(nn.Module):
    def __init__(self, model_name: str, num_intents: int, dropout: float = 0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.intent_head = nn.Linear(hidden_size, num_intents)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        output = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(output.last_hidden_state[:, 0])
        return self.intent_head(pooled)


# 5. Training
Load the data and train the model.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(config.seed)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
train_dataset = ModerationDataset("data/train.csv", tokenizer, config.max_length)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

model = IntentModel(config.model_name, num_intents=len(INTENT_MAP)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
loss_fn = nn.CrossEntropyLoss()

print(f"Training on {device}...")
model.train()
for epoch in range(config.epochs):
    total_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_id = batch["intent_id"].to(device)

        intent_logits = model(input_ids, attention_mask)
        loss = loss_fn(intent_logits, intent_id)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: avg loss {total_loss / len(train_loader):.4f}")

# Save the model
out_path = Path("saved_models/v1")
out_path.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_path / "model.pt")
tokenizer.save_pretrained(out_path)
print(f"Model saved to {out_path}")


# 6. Evaluation
Evaluate the checkpoint on `eval.csv`.

In [ ]:
model.eval()
eval_dataset = ModerationDataset("data/eval.csv", tokenizer, config.max_length)
eval_loader = DataLoader(eval_dataset, batch_size=config.batch_size)

gold_intents = []
pred_intents = []

with torch.no_grad():
    for batch in tqdm(eval_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        intent_logits = model(input_ids, attention_mask)
        intent_probs = torch.softmax(intent_logits, dim=-1)
        intent_best = intent_probs.max(dim=-1)

        gold_intents += batch["intent_id"].tolist()
        pred_intents += intent_best.indices.tolist()

labels = list(INTENT_MAP.keys())
target_names = [INTENT_MAP[i] for i in labels]

report = classification_report(gold_intents, pred_intents, labels=labels, target_names=target_names, zero_division=0)
print(report)


# 7. Test the Model
Try giving the model a custom comment.

In [ ]:
def classify_comment(text: str):
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=config.max_length,
        padding="max_length",
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        intent_logits = model(encoding["input_ids"], encoding["attention_mask"])
        intent_probs = torch.softmax(intent_logits, dim=-1).squeeze(0)

    intent_id = int(intent_probs.argmax())
    confidence = float(intent_probs[intent_id])

    return {
        "text": text,
        "intent": INTENT_MAP[intent_id],
        "confidence": round(confidence, 4),
        "abstain": confidence < 0.5
    }

# Test it
print(classify_comment("តំលៃប៉ុន្មានបង?"))
